# Long-Term Memory (the Store)

**What you will build:** an agent that remembers facts about a **user across different conversations** — not just within one thread. In 2.3 the checkpointer gave each `thread_id` its own memory; that's *short-term*. LangGraph's **Store** is *long-term*: it remembers a user even in a brand-new thread. This is the memory tier most "personal assistant" features actually need, and it's new versus everything before it.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/28_long_term_memory.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
!pip install -q "langchain>=1.3,<2.0" "langgraph>=1.2,<2.0" "langchain-openai>=1.3,<2.0"

import os, getpass
from langchain_openai import ChatOpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
llm = ChatOpenAI(model=MODEL_NAME, base_url="https://openrouter.ai/api/v1",
                 api_key=os.environ["OPENROUTER_API_KEY"])
print("Ready:", MODEL_NAME)

## Two tiers of memory

```
Checkpointer (2.3)   keyed by thread_id    remembers ONE conversation      short-term
Store (this)         keyed by user_id      remembers a USER across threads  long-term
```

You attach **both** when you compile: the checkpointer for the running conversation, the store for durable, cross-thread facts. A node reads and writes the store via the injected `runtime.store`.

In [ ]:
import uuid
from dataclasses import dataclass
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime

@dataclass
class Context:
    user_id: str          # who we're talking to — the key for long-term memory

def call_model(state: MessagesState, runtime: Runtime[Context]) -> dict:
    ns = ("memories", runtime.context.user_id)          # this user's namespace
    known = [m.value["data"] for m in runtime.store.search(ns)]   # everything we remember about them

    last = state["messages"][-1].content
    if "remember" in last.lower():                       # naive: store anything they ask us to remember
        runtime.store.put(ns, str(uuid.uuid4()), {"data": last})

    system = "What you already know about the user:\n" + ("\n".join(known) or "(nothing yet)")
    resp = llm.invoke([{"role": "system", "content": system}, *state["messages"]])
    return {"messages": resp}

builder = StateGraph(MessagesState, context_schema=Context)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

graph = builder.compile(checkpointer=InMemorySaver(), store=InMemoryStore())

## Prove it: remember in one thread, recall in another

Tell the agent a fact in conversation 1, then start a **brand-new** conversation (different `thread_id`) for the same user and see it recall the fact.

In [ ]:
user = Context(user_id="u-42")

# Conversation 1
graph.invoke({"messages": [{"role": "user", "content": "Please remember: my name is Bob and I love hiking."}]},
             config={"configurable": {"thread_id": "conv-1"}}, context=user)

# Conversation 2 — a DIFFERENT thread, no shared checkpointer state
r = graph.invoke({"messages": [{"role": "user", "content": "What's my name, and what do I like to do?"}]},
                 config={"configurable": {"thread_id": "conv-2"}}, context=user)
print(r["messages"][-1].content)   # recalls Bob + hiking, across threads

It remembered — even though conversation 2 is a fresh thread with no checkpointer history. The fact lived in the **Store**, keyed by `user_id`, not by thread. Switch to a different `user_id` and the memory is gone: that's how one app serves many users without their memories bleeding together.

Checkpointer = *this conversation*; store = *this person*. Real assistants need both.

## Beyond the magic word: extract facts with the LLM

The node above only stores something when the user literally says "remember" — and then it stores the *whole raw message*. Real assistants don't work that way. People state facts in passing ("I'm vegetarian", "I'm based in Berlin") without ever asking you to remember them, and you want the *fact*, not the sentence it arrived in.

So make the model do the extracting: on each turn, ask a cheap LLM call "is there a durable fact about this user here?" and store what it returns. This is the difference between a keyword trigger and actual memory formation.

In [ ]:
import json

def extract_facts(text: str) -> list:
    """Ask the LLM for durable facts worth storing — not the raw message, not a keyword match."""
    r = llm.invoke([{"role": "system", "content":
        "Extract durable facts about the user worth remembering long-term (identity, "
        "preferences, constraints). Reply with ONLY a JSON list of short strings. "
        "If there is nothing worth storing, reply []."},
        {"role": "user", "content": text}])
    raw = r.content.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        facts = json.loads(raw)
        return facts if isinstance(facts, list) else []
    except Exception:
        return []          # never let a malformed extraction crash the turn

def call_model_v2(state: MessagesState, runtime: Runtime[Context]) -> dict:
    ns = ("memories", runtime.context.user_id)
    for fact in extract_facts(state["messages"][-1].content):   # no magic word — the LLM decides
        runtime.store.put(ns, str(uuid.uuid4()), {"data": fact})
    known = [m.value["data"] for m in runtime.store.search(ns)]
    system = "What you already know about the user:\n" + ("\n".join(known) or "(nothing yet)")
    resp = llm.invoke([{"role": "system", "content": system}, *state["messages"]])
    return {"messages": resp}

b2 = StateGraph(MessagesState, context_schema=Context)
b2.add_node("call_model", call_model_v2)
b2.add_edge(START, "call_model")

store2 = InMemoryStore()                       # keep a handle so we can inspect it below
graph2 = b2.compile(checkpointer=InMemorySaver(), store=store2)

Now prove two things at once: a fact stated **in passing** (no "remember") survives into a fresh thread, and a **second user** sees none of it.

In [ ]:
alice = Context(user_id="alice")

# Thread A-1: Alice mentions facts casually — note there is NO "remember" anywhere.
graph2.invoke({"messages": [{"role": "user", "content":
        "Hi! I'm Alice, I'm vegetarian, and I'm planning a trip to Japan in spring."}]},
    config={"configurable": {"thread_id": "a-1"}}, context=alice)

print("stored for alice:", [m.value["data"] for m in store2.search(("memories", "alice"))])

# Thread A-2: brand-new conversation, same user — the facts come back.
r = graph2.invoke({"messages": [{"role": "user", "content": "Suggest one dish I'd enjoy on my trip."}]},
    config={"configurable": {"thread_id": "a-2"}}, context=alice)
print("alice ->", r["messages"][-1].content[:130])

# A DIFFERENT user: separate namespace, so none of Alice's memory leaks in.
bob = Context(user_id="bob")
r = graph2.invoke({"messages": [{"role": "user", "content": "What do you know about me?"}]},
    config={"configurable": {"thread_id": "b-1"}}, context=bob)
print("bob   ->", r["messages"][-1].content[:130])

Two results worth pausing on. First, Alice never said "remember" — the LLM extractor pulled *"vegetarian"* and *"trip to Japan"* out of a casual hello and stored them as clean facts, so the dish suggestion in the new thread respects both. Second, **bob** — a different `user_id`, hence a different namespace — knows nothing, because his memory search hits an empty namespace. That namespace boundary is the entire multi-tenant story: one store, many users, zero bleed.

The trade-off vs. the keyword version: extraction costs an extra LLM call per turn and can occasionally over- or under-store. That's the classic precision/recall dial of memory systems — worth it the moment you want memory that feels automatic rather than manual.

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Test extraction precision.** Send a turn with no durable fact ("What's the weather like?") and one dense with facts ("I'm lactose-intolerant, I use they/them, and I code in Rust"). Print `store2.search(...)` after each. **Done when:** the chit-chat stores nothing (`[]`) and the dense turn stores each fact separately — not one blob.
2. **Update, don't just append.** Alice says "actually, I eat fish now." The naive store keeps *both* "vegetarian" and the new fact. **Done when:** you can describe (or implement) a strategy — e.g. search existing facts and have the LLM merge/replace — and explain why unbounded append eventually poisons the context.
3. **Prove isolation is real, not luck.** Store a fact for `bob`, then re-run the `alice` recall. **Done when:** Alice's answer never mentions Bob's fact, and you can point at the exact line (`("memories", user_id)`) that guarantees it.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 —** extraction is per-message, so just call the node twice and inspect:

```python
for msg in ["What's the weather like?",
            "I'm lactose-intolerant, I use they/them, and I code in Rust."]:
    print(msg, "->", extract_facts(msg))
```

The first returns `[]` (nothing durable); the second returns three separate strings. Storing atomic facts — not whole sentences — is what makes later retrieval and updating tractable.

**2 —** Append-only memory is a slow leak: contradictory facts pile up and the system prompt grows without bound (the *context rot* problem from the coding-agents chapters). A real system reconciles on write — search the namespace, ask the LLM "does this new fact update or replace an existing one?", and `store.put` to the *same key* to overwrite (or `store.delete` the stale one). Sketch:

```python
existing = store2.search(("memories", "alice"))
# feed `existing` + the new fact to the LLM; have it return the reconciled set; rewrite the namespace
```

**3 —** Isolation isn't heuristic — it's the namespace key. `("memories", user_id)` gives each user a disjoint slice of the store; `bob`'s writes and `alice`'s reads never touch the same key, so there's nothing to leak. Change the namespace to a constant and every user would suddenly share one memory — the bug the Common issues table warns about.
::::

::::{dropdown} 🔍 Semantic long-term memory
:color: info

Here we retrieved **all** of a user's memories with `store.search(ns)`. For many memories you want the *relevant* ones — that's RAG (1.7) applied to memory. Give the store an embedding index and `search(ns, query=...)` returns the closest matches:

```python
from langchain.embeddings import init_embeddings
store = InMemoryStore(index={"embed": init_embeddings("openai:text-embedding-3-small"),
                             "dims": 1536, "fields": ["data"]})
# ... then: runtime.store.search(ns, query=last, limit=3)
```

`embed` also accepts a plain function, so you can plug in the keyless `sentence-transformers` embedder from 1.7 instead of a paid embeddings API.
::::

::::{dropdown} 🔧 Common issues (and fixes)
:color: secondary

| Symptom | Likely cause | Fix |
|---------|--------------|-----|
| **Doesn't recall across threads** | Only a checkpointer, no store | Compile with `store=...` and read it via `runtime.store` |
| **Users see each other's memories** | Shared namespace | Namespace by `user_id`: `("memories", user_id)` |
| **`runtime.store` is None** | Compiled without a store, or wrong node signature | Pass `store=` at compile; type the param `Runtime[Context]` |
::::

**What's next — Block 3:** you have all the pieces. Now we ship them — deploy an agent behind a real **API** (30), connect a **front end** (31), and build a **capstone project** (32).